<a href="https://colab.research.google.com/github/ERA-Software/computational-data-analysis/blob/main/notebooks/T6_from_training_to_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **From Training to Tuning - Week 8** 🔍 🤖 🐍

## Build an End-to-End ML Pipeline

**Duration:** 90 minutes | **Level:** Intermediate | **Format:** 3 open challenges + guided sections

---

## Context

Classification is one of the most common ML tasks. Today you will build a complete, reproducible workflow for predicting whether a tumour is malignant or benign — starting from raw data and ending with an honest test-set evaluation.

A production-grade ML pipeline requires:
- **Careful data splitting** before any preprocessing
- **Leak-free pipelines** that fit only on training data
- **Evaluation beyond accuracy** — confusion matrices, precision/recall, ROC curves
- **Learning curves** to diagnose whether more data or a better model would help most

---

## The Three Challenges

| Challenge | What you build | Time |
|-----------|---------------|------|
| **Challenge 1 — Section 5** | Preprocessing pipeline (imputation + scaling) | 20 min |
| **Challenge 2 — Section 7** | Evaluate the model: metrics + plots | 20 min |
| **Challenge 3 — Section 8** | Learning curves vs. training set size | 20 min |
| **Guided sections (1–4, 6, 9–10)** | Load data, EDA, split, baseline, final eval | 30 min |

---

## Objectives

By the end of this session you will be able to:

1. **Explain** what logistic regression does and why it is called a regression despite being a classifier
2. **Split data correctly** into train / validation / test sets before any preprocessing
3. **Build a leak-free preprocessing pipeline** with `sklearn.pipeline.Pipeline`
4. **Fit and evaluate** a logistic regression classifier with accuracy, confusion matrix, classification report, and ROC-AUC
5. **Plot and interpret learning curves** — accuracy vs. training set size for a fixed model
6. **Report final performance** on the held-out test set only after all decisions are made

---

## Self-Study Resources

### Core Reading

- **[Logistic Regression — scikit-learn User Guide](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression)** — Covers the model, solvers, and regularisation.
- **[Pipeline — scikit-learn User Guide](https://scikit-learn.org/stable/modules/pipeline.html)** — How to chain transformers and estimators safely.
- **[Learning Curves — scikit-learn User Guide](https://scikit-learn.org/stable/modules/learning_curve.html)** — Using training set size to diagnose bias vs. variance.

### API References

- **[LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)**
- **[Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)**
- **[ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html)**
- **[learning_curve](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.learning_curve.html)**
- **[confusion_matrix](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)** / **[roc_auc_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html)**

### Key Concepts

| Concept | What it does |
|---------|-------------|
| `train_test_split(X, y, test_size=..., stratify=y, random_state=42)` | Split data, preserving class proportions |
| `Pipeline([('step', transformer), ...])` | Chain preprocessing + model; `.fit()` only touches train data |
| `StandardScaler()` | Zero-mean, unit-variance scaling |
| `SimpleImputer(strategy='median')` | Fill missing values with training-set median |
| `LogisticRegression(C=1.0)` | Higher `C` = weaker regularisation |
| `learning_curve(estimator, X, y, train_sizes=...)` | Scores at increasing training sizes via cross-validation |
| `confusion_matrix(y_true, y_pred)` | Counts of TP, FP, FN, TN |
| `roc_auc_score(y_true, y_proba)` | Area under the ROC curve (1.0 = perfect) |

---
## Setup — Imports

In [ ]:
# Run this cell first — all imports for the entire notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

pd.set_option('display.max_rows', 20)
%matplotlib inline

print("Libraries loaded successfully!")

---
## Section 1 — Introduction & Problem Framing

### What is Logistic Regression?

**Logistic regression** is a linear classifier that estimates the **probability** that an observation belongs to a class.
Despite the word *regression*, it is used for **classification**: the output is a probability $p \in (0, 1)$, and we threshold at 0.5 (or another value) to obtain a class label.

The name comes from the **logistic (sigmoid) function** used to squash the linear combination of features into a valid probability:

$$p(y=1 \mid \mathbf{x}) = \sigma(\mathbf{w}^\top \mathbf{x} + b) = \frac{1}{1 + e^{-(\mathbf{w}^\top \mathbf{x} + b)}}$$

The model is *linear in the log-odds* — hence the word regression in the name — but the output is a *classification decision*.

### When to Use Logistic Regression

| Use it when… | Avoid it when… |
|---|---|
| The decision boundary is (roughly) linear | Classes are highly non-linearly separable |
| Interpretability matters (coefficients = feature weights) | You need to capture complex interactions without feature engineering |
| Data is relatively small or high-dimensional | You have massive data and a deep architecture is affordable |
| You want calibrated probabilities | You only care about hard class labels |

Logistic regression is an excellent **baseline**: if a fancier model cannot beat it, question whether the extra complexity is worth it.

### Regularisation (`C` parameter)

scikit-learn's `LogisticRegression` adds an L2 penalty by default.  
The parameter `C = 1 / λ` controls regularisation strength:
- **Small `C`** → strong regularisation → simpler model → higher bias, lower variance (underfitting risk)
- **Large `C`** → weak regularisation → complex model → lower bias, higher variance (overfitting risk)

We will tune `C` in Section 8 using a validation set.

---
## Section 2 — Load Data

We use the **Breast Cancer Wisconsin** dataset: 569 samples, 30 numerical features, binary target (malignant vs. benign).

We use `sklearn.datasets.load_breast_cancer()`.

In [ ]:
from sklearn.datasets import load_breast_cancer
raw = load_breast_cancer(as_frame=True)
X = raw.data
y = pd.Series(raw.target, name='target')  # 0 = malignant, 1 = benign

print(f"\nFeature matrix : {X.shape}  (rows=samples, cols=features)")
print(f"Target vector  : {y.shape}")
print(f"Classes        : {sorted(y.unique())}")
print(f"Class counts   :\n{y.value_counts()}")
X.head(3)

In [ ]:
# Inspect feature types and missing values
print("Feature dtypes:")
print(X.dtypes.value_counts())
print(f"\nMissing values per column (total): {X.isnull().sum().sum()}")
print("\nSummary statistics (first 5 features):")
X.describe().iloc[:, :5].round(2)

---
## Section 3 — Exploratory Data Analysis

> **EDA uses the full dataset intentionally.** We are studying the data's structure and distribution to inform modelling choices (which features matter? are there class imbalances? are features correlated?). This is different from fitting a model — we are not estimating any parameters that could leak into evaluation. The actual preprocessing (scaling, imputation) will be fitted only on the training set in Section 5.

We reuse the same visualisation patterns as T4: histograms, colour-coded scatter, and a correlation heatmap.

In [ ]:
# ── Class balance ────────────────────────────────────────────────────────
counts = y.value_counts().sort_index()
labels = [f'Class {c}' for c in counts.index]

plt.figure(figsize=(5, 4))
plt.bar(labels, counts.values, color=['steelblue', 'darkorange'], edgecolor='black')
plt.ylabel('Number of samples')
plt.title('Class Balance')
plt.tight_layout()
plt.show()

print(f"Class ratio: {counts.values[0] / counts.values[1]:.2f}  (class 0 / class 1)")

In [ ]:
# ── Feature distributions — first 6 features, coloured by class ──────────
feature_subset = X.columns[:6]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))

for ax, feat in zip(axes.flat, feature_subset):
    for cls, color, lbl in [(0, 'steelblue', 'Class 0'), (1, 'darkorange', 'Class 1')]:
        ax.hist(X.loc[y == cls, feat].dropna(), bins=25,
                color=color, alpha=0.6, edgecolor='none', label=lbl)
    ax.set_title(feat, fontsize=9)
    ax.set_ylabel('Count')
    ax.legend(fontsize=7)

plt.suptitle('Feature Distributions by Class (first 6 features)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation heatmap — first 10 features ───────────────────────────────
# Highly correlated features carry redundant information.
# This motivates StandardScaler (removes scale differences) and possibly PCA as an extension.
corr = X.iloc[:, :10].corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label='Pearson r')
ax.set_xticks(range(len(corr)))
ax.set_yticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(corr.columns, fontsize=8)
ax.set_title('Feature Correlation Matrix (first 10 features)')
plt.tight_layout()
plt.show()

# Identify pairs with |r| > 0.9
high_corr = [(corr.columns[i], corr.columns[j], corr.iloc[i, j])
             for i in range(len(corr)) for j in range(i+1, len(corr))
             if abs(corr.iloc[i, j]) > 0.9]
print(f"Highly correlated pairs (|r|>0.9): {len(high_corr)}")
for a, b, r in high_corr[:5]:
    print(f"  {a}  ↔  {b}  r={r:.3f}")

---
## Section 4 — Train / Validation / Test Split

### Why split *before* preprocessing?

When you compute a mean, standard deviation, or median over the **full** dataset and then use those statistics to scale/impute, you have effectively leaked information from your validation and test sets into training.
The model "sees" statistics it would never have at inference time — this artificially inflates your performance estimates.

**The rule:** every transformer (`StandardScaler`, `SimpleImputer`, `PCA`, target encoders, feature selectors) must be `.fit()` on training data only. Then `.transform()` is applied to validation and test sets using training statistics.

### Why three sets, not two?

| Set | Purpose | Touched when? |
|-----|---------|---------------|
| **Train** | Fit model parameters | Every iteration |
| **Validation** | Tune hyperparameters, detect overfitting | During model selection |
| **Test** | Report final unbiased performance | Once, at the very end |

If you use the test set to make any modelling decision (which `C` to pick, which features to keep), it is no longer held-out — your reported performance will be optimistic.  
The validation set absorbs all those decisions so the test set stays clean.

We create an **80 / 10 / 10** split with `stratify=y` to preserve class proportions in each partition.

In [ ]:
# Step 1: carve out the test set (10% of total)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size=0.10,          # 10% held out as test
    stratify=y,              # preserve class ratio in both halves
    random_state=42
)

# Step 2: split the remaining 90% into train (80%) and validation (10%)
# 0.10 / 0.90 ≈ 0.111 gives us ~10% of the total as validation
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.111,
    stratify=y_trainval,
    random_state=42
)

print(f"Train      : {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Validation : {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.0f}%)")
print(f"Test       : {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)")

# Confirm class proportions are preserved
for name, yp in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    print(f"  {name} class-1 fraction: {yp.mean():.3f}")

---
# 🛠️ **CHALLENGE 1 — Section 5: Preprocessing Pipeline**

---

## Your Task

Build a preprocessing pipeline that:

1. **Handles missing values** — impute with the column median
2. **Standardises numerical features** — zero mean, unit variance

The pipeline must be **fitted on `X_train` only**, then used to transform `X_val` and `X_test`.
Do **not** call `.fit_transform()` on the validation or test sets.

After building and applying the pipeline, you should have:
- `X_train_proc` — preprocessed training features
- `X_val_proc`   — preprocessed validation features (using training statistics)
- `X_test_proc`  — preprocessed test features (using training statistics)

---

### Hints / Scaffolding

- Use `sklearn.pipeline.Pipeline` to chain `SimpleImputer` → `StandardScaler`.
  ```python
  from sklearn.pipeline import Pipeline
  from sklearn.preprocessing import StandardScaler
  from sklearn.impute import SimpleImputer
  ```
- For datasets with **mixed types** (numerical + categorical), use `ColumnTransformer` to apply different transformers to different column groups.
  This dataset is all-numerical, so a single `Pipeline` is sufficient.
- Fit with `.fit(X_train)`, then transform with `.transform(X_train)`, `.transform(X_val)`, `.transform(X_test)`.

```python
def build_preprocessor(numerical_cols):
    """
    Build and return an unfitted preprocessing pipeline.
    numerical_cols: list of column names (or all columns for an all-numeric dataset)
    """
    # YOUR CODE HERE
    pass
```

---

> ⚠️ **Things to watch out for**
> - **Never call `.fit_transform()` on `X_val` or `X_test`** — this recomputes statistics from those sets, which is data leakage.
> - **Never compute any global statistics (mean, std, median) before the split.** Wait until you have `X_train`, then fit your scaler/imputer on that.
> - If your scaler returns a NumPy array, downstream cells expect that — no need to convert back to a DataFrame.

In [ ]:
# ── YOUR SOLUTION ────────────────────────────────────────────────────────
# Complete the function below, then fit and transform the three splits.

def build_preprocessor(numerical_cols):
    """
    Build and return an unfitted preprocessing pipeline.
    numerical_cols: list of column names (or all columns for an all-numeric dataset)
    """
    # YOUR CODE HERE
    pass


# YOUR CODE HERE


print("Preprocessing done.")

### Reference Solution — run this cell if stuck

The cell below provides a complete implementation. Run it to ensure the rest of the notebook executes end-to-end.

In [ ]:
# ✅ REFERENCE SOLUTION — Section 5

def build_preprocessor(numerical_cols):
    """
    Returns an unfitted pipeline: median imputation → standard scaling.
    Works for any list of numerical columns.
    """
    numeric_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),  # fill NaNs with column median
        ('scaler',  StandardScaler()),                  # zero mean, unit variance
    ])
    # ColumnTransformer lets us extend to mixed-type datasets in the future
    preprocessor = ColumnTransformer(
        transformers=[('num', numeric_pipe, numerical_cols)],
        remainder='drop'
    )
    return preprocessor

# Build, fit on train, transform all three partitions
preprocessor = build_preprocessor(X_train.columns.tolist())
preprocessor.fit(X_train)                           # statistics computed from train only

X_train_proc = preprocessor.transform(X_train)      # apply same transform to each split
X_val_proc   = preprocessor.transform(X_val)
X_test_proc  = preprocessor.transform(X_test)

print(f"X_train_proc shape : {X_train_proc.shape}")
print(f"X_val_proc shape   : {X_val_proc.shape}")
print(f"X_test_proc shape  : {X_test_proc.shape}")
print(f"Train feature mean (should be ~0): {X_train_proc.mean(axis=0)[:3].round(4)}")
print(f"Train feature std  (should be ~1): {X_train_proc.std(axis=0)[:3].round(4)}")

---
## Section 6 — Fit Baseline Model

We fit a `LogisticRegression` with default settings (`C=1.0`, L2 regularisation) on the preprocessed training data.  
This gives us a **baseline** to compare against when we tune `C` in Section 8.

In [ ]:
# Fit logistic regression with default C=1.0
clf = LogisticRegression(C=1.0, max_iter=10000, random_state=42)
clf.fit(X_train_proc, y_train)

# Quick sanity check: training accuracy
train_acc = clf.score(X_train_proc, y_train)
print(f"Training accuracy (baseline, C=1.0): {train_acc:.4f}")

---
# 🛠️ **CHALLENGE 2 — Section 7: Evaluate the Model**

---

## Your Task

You have a fitted baseline classifier (`clf`, `C=1.0`) and preprocessed validation data (`X_val_proc`, `y_val`).
Compute and visualise the following:

1. **Accuracy** on the validation set
2. **Confusion matrix** — display with `ConfusionMatrixDisplay`
3. **Classification report** — precision, recall, F1 per class
4. **ROC-AUC score** and **ROC curve** (True Positive Rate vs. False Positive Rate)

---

### Scaffolding

```python
# Predictions
y_val_pred  = ...   # hard class labels  →  clf.predict(...)
y_val_proba = ...   # P(class 1)         →  clf.predict_proba(...)[:, 1]

# Metrics
val_acc = accuracy_score(y_val, y_val_pred)
val_auc = roc_auc_score(y_val, y_val_proba)

# Confusion matrix
fpr, tpr, _ = roc_curve(y_val, y_val_proba)
```

---

> **Tips**
> - `clf.predict(X_val_proc)` returns hard class labels (0 or 1).
> - `clf.predict_proba(X_val_proc)[:, 1]` returns the probability of class 1 — needed for ROC-AUC.
> - `roc_curve(y_val, y_val_proba)` returns `(fpr, tpr, thresholds)`.
> - Plot both the confusion matrix and the ROC curve side by side with `plt.subplots(1, 2)`.

In [ ]:
# ── YOUR SOLUTION ─────────────────────────────────────────────────────────

# TODO: generate predictions from clf
y_val_pred  = None  # replace with clf.predict(X_val_proc)
y_val_proba = None  # replace with clf.predict_proba(X_val_proc)[:, 1]

# TODO: compute accuracy and ROC-AUC
val_acc = None
val_auc = None

# TODO: print val_acc, val_auc, and classification_report(y_val, y_val_pred)

# TODO: plot confusion matrix and ROC curve
print("Challenge 2 — fill in the code above.")

In [ ]:
# ✅ REFERENCE SOLUTION — Section 7

# Predictions on validation set
y_val_pred  = clf.predict(X_val_proc)
y_val_proba = clf.predict_proba(X_val_proc)[:, 1]  # probability of class 1

val_acc = accuracy_score(y_val, y_val_pred)
val_auc = roc_auc_score(y_val, y_val_proba)

print(f"Validation accuracy : {val_acc:.4f}")
print(f"Validation ROC-AUC  : {val_auc:.4f}")
print()
print(classification_report(y_val, y_val_pred, target_names=['Class 0', 'Class 1']))

# ── Confusion matrix + ROC curve ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_val, y_val_pred),
    display_labels=['Class 0', 'Class 1']
).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix — Validation Set')

fpr, tpr, _ = roc_curve(y_val, y_val_proba)
axes[1].plot(fpr, tpr, color='steelblue', linewidth=2,
             label=f'Logistic Regression (AUC = {val_auc:.3f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1, label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — Validation Set')
axes[1].legend()

plt.tight_layout()
plt.show()

---
# 🛠️ **CHALLENGE 3 — Section 8: Learning Curves**

---

## What Are Learning Curves?

A **learning curve** plots model performance (train and validation accuracy) as a function of **training set size**, with the model architecture and hyperparameters held **fixed**.

They answer a key diagnostic question: *does this model benefit from more data?*

| Pattern observed | Diagnosis | What to try |
|-----------------|-----------|-------------|
| Both curves low and close together | **High bias** — model too simple | More features, less regularisation, different model |
| Large train–val gap at all sizes | **High variance** — model too complex for dataset size | More data, stronger regularisation, simpler model |
| Val curve still rising at the right edge | Model has not saturated — **collect more data** |
| Val curve has plateaued / converged | More data won't help — focus on the model itself |

Unlike a regularisation sweep (which fixes data and varies the model), learning curves **fix the model** and vary how much data it sees.
This isolates the data-quantity bottleneck from the model-capacity bottleneck.

---

## Your Task

Fix the model at `C=1.0` and plot a learning curve over training sizes from 10% to 100% of `X_train_proc`.

1. Use `sklearn.model_selection.learning_curve` to obtain train and CV-validation scores at each size.
2. Plot **mean ± 1 std** for both train and validation accuracy vs. number of training samples.
3. Interpret: is this model limited by data quantity or model capacity?

---

### Scaffolding

```python
from sklearn.model_selection import learning_curve

train_sizes_abs, train_scores, val_scores = learning_curve(
    estimator    = ...,                        # LogisticRegression(C=1.0, ...)
    X            = X_train_proc,
    y            = y_train,
    train_sizes  = np.linspace(0.1, 1.0, 10), # fractions → converted to absolute counts
    cv           = 5,                          # 5-fold cross-validation at each size
    scoring      = 'accuracy',
    random_state = 42,
)
# train_scores / val_scores  shape: (n_sizes, n_cv_folds)
# Average across folds: train_scores.mean(axis=1)
```

Shade uncertainty with:
```python
plt.fill_between(train_sizes_abs,
                 mean - std, mean + std,
                 alpha=0.15, color='steelblue')
```

---

> ⚠️ **Things to watch out for**
> - `learning_curve` runs its own internal cross-validation — you do not manually loop over sizes.
> - The function returns `train_sizes_abs` (absolute sample counts), not the fractions you passed in.
> - Keep the model fixed (`C=1.0`) — you are not tuning hyperparameters here.

In [ ]:
# ── YOUR SOLUTION ─────────────────────────────────────────────────────────

# TODO: define your fixed model (C=1.0)
model_fixed = None  # LogisticRegression(C=1.0, max_iter=10000, random_state=42)

# TODO: call learning_curve with train_sizes=np.linspace(0.1, 1.0, 10), cv=5
# train_sizes_abs, train_scores, val_scores = learning_curve(...)

# TODO: compute mean and std across CV folds (axis=1)
# train_mean = ...  train_std = ...
# val_mean   = ...  val_std   = ...

# TODO: plot the learning curve
#   - mean line for train (steelblue) and val (darkorange)
#   - shaded ±1 std band
#   - label axes and add a legend

print("Challenge 3 — fill in the code above.")

In [ ]:
# ✅ REFERENCE SOLUTION — Section 8

# Fixed model — we are not tuning C here, only varying training set size
model_fixed = LogisticRegression(C=1.0, max_iter=10000, random_state=42)

train_sizes_abs, train_scores, val_scores = learning_curve(
    estimator    = model_fixed,
    X            = X_train_proc,
    y            = y_train,
    train_sizes  = np.linspace(0.1, 1.0, 10),
    cv           = 5,
    scoring      = 'accuracy',
    random_state = 42,
    n_jobs       = -1,
)

# Aggregate across CV folds
train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

print(f"Train acc  — smallest subset : {train_mean[0]:.4f}  |  full training set : {train_mean[-1]:.4f}")
print(f"Val   acc  — smallest subset : {val_mean[0]:.4f}  |  full training set : {val_mean[-1]:.4f}")
gap = train_mean[-1] - val_mean[-1]
print(f"Train-val gap at full size   : {gap:.4f}  ({'high variance' if gap > 0.05 else 'low / well-fit'})")

# ── Plot ──────────────────────────────────────────────────────────────────
plt.figure(figsize=(9, 5))

plt.plot(train_sizes_abs, train_mean, color='steelblue', linewidth=2,
         marker='o', markersize=5, label='Training accuracy')
plt.fill_between(train_sizes_abs,
                 train_mean - train_std, train_mean + train_std,
                 alpha=0.15, color='steelblue')

plt.plot(train_sizes_abs, val_mean, color='darkorange', linewidth=2,
         marker='s', markersize=5, label='CV validation accuracy')
plt.fill_between(train_sizes_abs,
                 val_mean - val_std, val_mean + val_std,
                 alpha=0.15, color='darkorange')

plt.xlabel('Training set size (number of samples)')
plt.ylabel('Accuracy')
plt.title('Learning Curves — LogisticRegression (C = 1.0, fixed)')
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## Section 9 — Final Evaluation on the Held-Out Test Set

> **This is the one and only time we touch the test set.** All modelling decisions — which preprocessing pipeline, which model, `C=1.0` — were made using the training and validation data in earlier sections. Only now do we apply the final model to `X_test_proc` and report an unbiased estimate of real-world performance.

We use `C=1.0`, the fixed model from our learning curve analysis. The learning curves showed whether data quantity is the bottleneck, not whether `C` itself needs tuning — that would be a separate hyperparameter search exercise.

In [ ]:
# Fit final model with C=1.0 on training data
clf_final = LogisticRegression(C=1.0, max_iter=10000, random_state=42)
clf_final.fit(X_train_proc, y_train)


# TODO: evaluate on test set
# y_test_pred  = ...
# y_test_proba = ...

# TODO: compute test accuracy and ROC-AUC
# test_acc = ...
# test_auc = ...

# print("=" * 45)
# print("  FINAL TEST SET RESULTS")
# print("=" * 45)
# print(f"  Accuracy : {test_acc:.4f}")
# print(f"  ROC-AUC  : {test_auc:.4f}")
# print()
# print(classification_report(y_test, y_test_pred, target_names=['Class 0', 'Class 1']))

# TODO: plot confusion matrix and ROC curve
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# YOUR CODE HERE

plt.tight_layout()
plt.show()

In [ ]:
# ✅ REFERENCE SOLUTION — Section 9

# Fit final model with C=1.0 on training data
clf_final = LogisticRegression(C=1.0, max_iter=10000, random_state=42)
clf_final.fit(X_train_proc, y_train)

# Evaluate on test set
y_test_pred  = clf_final.predict(X_test_proc)
y_test_proba = clf_final.predict_proba(X_test_proc)[:, 1]

test_acc = accuracy_score(y_test, y_test_pred)
test_auc = roc_auc_score(y_test, y_test_proba)

print("=" * 45)
print("  FINAL TEST SET RESULTS")
print("=" * 45)
print(f"  Accuracy : {test_acc:.4f}")
print(f"  ROC-AUC  : {test_auc:.4f}")
print()
print(classification_report(y_test, y_test_pred, target_names=['Class 0', 'Class 1']))

# Confusion matrix and ROC curve
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, y_test_pred),
    display_labels=['Class 0', 'Class 1']
).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix — Test Set')

fpr_t, tpr_t, _ = roc_curve(y_test, y_test_proba)
axes[1].plot(fpr_t, tpr_t, color='steelblue', linewidth=2,
             label=f'Logistic Regression (AUC = {test_auc:.3f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1, label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — Test Set')
axes[1].legend()

plt.tight_layout()
plt.show()

> [!NOTE]
> **Now take a break and look back:** outline the end-to-end data analysis pipeline, including the use of machine learning techniques for classification and regression.

---
## Section 10 — Wrap-Up

### Key Takeaways

| Concept | What you practised |
|---------|-------------------|
| **Logistic regression** | Linear classifier that models log-odds; interpretable coefficients |
| **Split before preprocess** | Prevents data leakage; scalers/imputers fitted on train only |
| **Three-way split** | Train/val/test: val absorbs all tuning decisions; test used once |
| **Evaluation beyond accuracy** | Confusion matrix, precision/recall, F1, ROC-AUC |
| **Learning curves** | Accuracy vs. training set size for a fixed model — diagnoses whether the bottleneck is data quantity or model capacity |
| **Final model** | Refit with chosen `C`, evaluate on test set exactly once |

---

### Extensions to Explore

1. **Tune `C` with cross-validation.** Use `sklearn.model_selection.GridSearchCV` or `RandomizedSearchCV` to select the best regularisation strength. How does the optimal `C` compare to the default of 1.0?

2. **Compare learning curves across models.** Plot learning curves for `LogisticRegression`, `RandomForestClassifier`, and `SVC` on the same axes. Which model benefits most from additional data? Which plateaus earliest?

3. **Add polynomial features.** Insert `PolynomialFeatures(degree=2)` into your preprocessing pipeline. How do the learning curves change? Does the model become more data-hungry?